# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdfYt24zaS6K8g7puxlMiy3a9JlHV23ba62xM/ei27e+dYXh5KgiTGEqnhw26N1/9+6wGQAAlKcid7755lZtoiiUehqlBVqCqAj1vDKBwHk/ZiudURW8F8EcWpiJJ+qH7GMv+ZZcEov0mDefEmmWZpMCveTWPpj4Jw0g/HcTQX6XIBN0K9vVikQRT6s5Y4DZK0JY6DIfx7GC5V6YWfTmfBQBf/BLf9sB++EMcylcNUdMP7II7CuQzTjvgQRZOZFEfRzB+I+0ScRkMfADnpeUcXp4fvxAGMpY0ttuVX6C1p9Ld2YcQpVO5vNYUfjkQYpdVCR50+XltN7Hokx4CHJJrdS28UPYSzyB/J2BsFcaMpdn4VSRp3+qGAq79F/9Hv42XozwMAZ7YUMwArlYkG9zgO7qUomhLQFAwtipctkWQLHDciLJnCj2GWJi1usa0f7KR+PJFpsjNY7gQjhi1DrLZoRMnUj+VIjKMZtJ2Ih2kAfU4yP/Zh4BJbHsOThBuFouK8+7l7KR7iIAXEKGAA6jSyAY6jKG1XBxqMCYca5woVeA38RCKaDDL4gwT/NvQ9vAU0yYbnIUye12wW1WOZZnGYV/09CsKGbrIFIBi0SFJ/ggzHBMPKL8R+W3yKg7kfL8VwKod3nVoENqxh6kJCFRJRyAymQFOPPUQHjKzgqN0R1t+t6cVAV4ndzAabBvrSeGnc4TWOYph5IxEQVmZQG5mwrn7ejoe9GUQgTJq1Wthqs1pTkVbXCxLqjppzdYQX4iIIM+kAozIcfQHu4gCmx4G4ke1Z9CBxYuFYZXmk3PWtuxmA1g+XjTuspJvEVujBDfCLn/r9LeSc2H9QP+Q4CCXQ5rYpoCRWl20ZjpKHAJi0v9UOFstwgKIiB0e1XDd+vBTjMrTuYkk2qBCEituMTQxdM9YSF0GLLNBsauHj9aBCqWoZ+XUoFyBw6Q9Il40pvqbiwk+SYpq+bIsjnJ5i6IdRiAITf40CoJbMBWOgxVX+irilaLcyC8+WNJt3TVy2nlVe7Iiemsmrq/ZI3tLvxOqP6yhuRQYaIgMVQzDwUqXnsEw29/wBVTszOAlvScAO3dOZiptconrD5y72oec1hDdg1mKdm0GZgVOu/LYO6cDlrUpT5cJQ6HY9IyMAdqENuJgZsmZojIcKNYw+hwUzv2qLnvTj4VS4R0pcAFJGKWcwHXzW9ShvxHbBPNvc4nxJ1V1qJsddnVpRdddrFJRoy5KkdVTOq2SzmbcoSy9VocWtuZmPXmkB36axa0FribwaMtcrECaGeyIwuDViVAH2Z84Ko9n/ObOjQFLNLNHXhjIfr+qscQyd8aVosH76csFv1CSv88nHEpntOQVjYghp13QyhXj9nDJb+baJVddCXs85u8xaf8oUczD1aiLVEEhfYEBUoebCG1g0TmsGYHHbM2uBpDLfyETHcuxns1SMYeE28C2zpLDm1GpwtEIuOwwBo1ZVMvCrZs54eeH/DlHhLFQSIc8fox6lEjFcHR8fX3w5P704PO5eescnl9Bg3Wq6H8LISwvJ04ujw1Pvy8Xlb71Ph0ddG5xUzhfYtZwlUpV/d9jrqm6eu+isdmWxs24ZGTp3IuANQ9FkN0UvjbMhoAAwZy0rj01T9vLwSwlG6sFGVL5SgZYvu+9PzrvHG1bSqxqj4tnF55Nuz1XfaBorz6N7BNGse/V5g3rpvZfIuFz14upjTvVVtaN0qmTD8eHV4WbD5PUcVDnrXh1Stb/1Ls4r9VR7NDaZ+lir/XsShTnFuoupnMsYZji5j8TnM5tYzBY1JCvxjKKZF3DzqiqNddPaSDzty7i6POx99HpXhx9Ozj9s2EAa+8nU8ohwoTrEOprIMcvvVuLXbtyN5avu2SfVdak3oPflyWeNG5vGyu0H+iAOBlmKE+rTtTiaZUkKpvNfYNmYhXcJTDgaqU2zFSirMpKJq6OP1+e/9Vah3HhFkoCgMLClWtgM00bto9Pr3pV7spjYHfL4rSrYZPfS2afRKtZ+iOI7Gdsdnh3Sn/cnpxWRV6o997Hrgq5qwPWAV5nDBN+uvmIQlX7KQ7ELrBiQs6XysMpsp1nuiHzlWeyjFaFYxTu+vjy8Ork493rdI+jt5eu99l5uW4BRKuZBmKG/YgEt3MnlOAbVIxI5QQ+2aERgksxB6Az8mR8OpYjGgg2QNJqBTMJHfxFgzykt8v661xUnuxeaU73TLmqlq5Oz7sX1lYLh1Z6C4YV4k3c/k34ixUhBLxqLWN4DBAlaOvDCH4CtxW71BEvhjHq9M0d/BbIoWLh+mAyjEVit/ZBJ5X3sHl5eveseXpX6f/0Gun8hzvz4TjCVYFjjGegk9iiKKZjp6UD6KZnJr9/AhD07/A/v8OgKZYGmzt8u3vUQowUyj9jVFGWAzWAhscWOgIHsDLLxGH3dofyait+jwXbCYCciiUhqKGaBAve47A5gCTaaSSb1ZxkPIkDBaTQhQfIXMAIH2UScR2kwBssP8ZWIK3iLNY67764/eKcXH3odMYgiXHOiYSE5LNGeyBSMqe754btTkGp5WTYU9tHy/g4tmD1itN7fYaBnWICYtYOhhFVzP1kCH87BvJ8UfAqCFp3iFChJOrbNoSMiNHwc2jBaLLXI/ffr7jWA+G5Vh//IZCbbo4ExyeqqVae6VRlhPVGO9JbwZw/+MhEZwD0jAHu9Y9H799MglRScwNgDkBfYD72A1phoBmCZHahJgxr7wQzsreQXQVbEQwCtYsv2MF22pTEWe3CWRWmUKjdJ5kOYQO/AWUOZJBhtMJyWgpYz/RDZnE15FRwq6SBlXMAP0yyq2m7KDWkbZkXB3NpqiYIQlr6qqjd4ZkpDhzRuOa3jlrDsovw2t3VaomK96EIFcGWVmT8xYarVE/1Qr1jsBTgw5ty/k0CIpDFqMRG86O7gKs6ksvVXLAr1gpDkzTCLYxkOl6In5/5iGgGfgYD4wlLtFDg0VcJLk1TDh3KY35xefPF6n7pI1WqZPZRre8Aao4BjhNfnpydnJ1fdY3Ssc+8pKoCdZCFBF+klk+pVobtor0aOUhxTVQU7XQ0FyufB2nY+wIZrOIA2AMIjIDxnS+cRyHcYDP7tiCzE6Yvac2ggEedBgoMBZQeTPA6GdzNpjgkKg4BOgCAbQWkPX4mZq2Auoywl50GPkPbFT4fTUTQRlxkFPvPBAW9elrXXz6A8jSIF9a4+XnZ7Hy9Oj713nxDV+3t74gf49+XrNlER7397t5u4K1t97L+kTixEMyyX3atLmOys91BkBmkA8vFHuIWVtQqTvQL5mKLLMsWlZ5po8bozl/MIJNCVn9yJq9hX8hGQjh6/HQzaixkK0XkE7UYxxehTKEzRlFv120PBaiEdlMgdrs7xtTeMshCtIeBd7vfDLBooHQOPcWUcR7PE4Dd+4/nDlH3ZOA37Ia9SN3pZCxAH5xMfsfRPSct5WtnjP6RNnQH6niqfCF2BozNxlCQ7i5mfwt0cBjIH7RYMglmQLomZ7DCxP5bpsu2IhFPnlTB2f+s+GMlIl6Ve0RPSTrJBI96+wXyD3R/+tdPf+pdf/+t2u0Wwtqhc012l309+pHIiL9hGo3XRaFquGHxz0wGWu0UIYcCEnqb4FdlQoKKjIhqbgyUKQI+XkSM5k6lskJ8Edb0HE5N8QAV20Qhy5D+8o2bKSQ9X2Kp4F2h5exahPxbUNqt9ygCxKgQhmAL4fqbX5jJfriN3JAt/iJXjJFV6CgqHaGzPQcSEmMcQw6TAXoKUG5kt2+JqGiQiN4FN2P6RwdRCLk6yOWmG+pSHKlYwiOxIKKkWbFYZ5D3a4eu0mUOlurQbXlkYgAXmKa4Za8fAI+bxtPGf141meyq/3nR+un3yHjXIaEzSHHIA/aQRgBfhEhgkScumoANIAxgDxBdEf1wc2ESPkOhMcEpVIbO3qMb5Rm2kqgPKlgGZ1dc1qCRYyRCPceOjILlbFYM1Gio5jaEgs1XD2ZkiKcsyl7khfLANTZsDZHHaACLdEBfy5LkFxRWjL6gDlkg241wlnp6CiEli67GKgqeOeJRPlqu9GqGoDnc1m1aH7ihfGyQtUOFCh9MlX2pBzQ8WUkNYzobZgnulHBYP5JG/JBUARl/+uJMnnN1AgVtlpzQdAqs3hBVuNeNJNNzOb04JoU4FLHJw5pMHwUquGg5B5oPaxCQwnV+FaxQJZCwSjYoELhEkSQaiChqjdc5kGsH0glUb6H0l3ijDC3SPIdigxTmseFDXO4S3W34VGII+2XbT2XCOTC4mQTkmbbQBAG8QoXaIRkegWXfmkoVFqpAjWL3WsrdbfoFrZrBkwsQg5UivpxEb2pNp0pTZbKFrWlkooJrRaoDCjbi/9Z+8AP6h3wf+aP/wf1BLQ4mTD+cXl92jw1632aqtmvtQsTay1ar6t0b4CqEFvZbI+J4dPQM5RhtaTZiOLngyzpeIYH8DRpE2Ux/oqpftSFPLlYu9J8NMkjoeoNwvdDYygYteKBe9USXXqdZNoOu5mUW1VpZKNP2rneQpCTV9qH7KMX9uzI49YqYcGIzFy1/Fnls05pq6GGKdgtaXUmfomxkT/bmTlkaeo0qhLo6YrrfikmgzEvf+LBgVRCTqGNIMFOujavfJSJfMaYUUf14IZQOSYSUn0Vy9GWSr72wF7aizeurx6/92+mE3rQKl30JDa/T1hMRiha7f3NDIe1VGBmfJFgtGmuok+AprgtshWUKR4T1+YCU7KBFt9Ml5AI4c1NqkoUriKSE2b6fMS9UURIQISiI8tsCuGkHwBqieDqcNhsKVIOYiFQZKMUub1AaRBHO5xrCQRXxhU08VlsXLub7KR1Ytr7H944HYd7QGS+K7XMHqsjZ7O8A/UgWzhXhUlZ60BhzsFnaMY0XGaeDap3uZhTx4pWAwW3oejTIorjL4MZkhC4nDAuUAdvli11h00B2ov61ZNJnIuHa7AqwU3FsUMN/SerB6k0Jpa0IrNyKVh4kq76AZBisJ5IEo9icYqVHej37owfOq3+JSOy6A8INs4sFKazCTozySgBIGtXOXn6OVONIJKTjH/AQfQE0MVLQwNwUUMYgCDE2McB0FLcdyCNyH/cPExYHc4DjQ+qWx3N4qX4/HvskP3seT3tXF5d85cKQgz017t/2ce15kyiNpWCOxzWseDJoTsN6in/cq4kJVEYMUd/HJASPAWFYclFjelQm7mWzM8SvKnswxbnC+XRiAR+g0rMZcKwI4N87gzS2lnez3t5CbS42SA4UiOitXW8xYvN2mIr1wC07RH3TmGiVeG+YyIW2CRJHG7aW5JFs4YZaDUblpAmsDBYNFC7UkK0NZMIXmoYaDh2zu0K9gPZPgbgsffTcJsC36vcnAJM2B7I8sLRqyPWmjlDGFUtPJKRYjr+MUqzBQYKE2HuGIPHQI+qmHEiRJ/fmivPFH4UNLmXb+I4weKA8vHuNto7/1/cfO92ed73vG3iIPRV/socspgv4xqtmwcfQlWIAoxrzhhQxRkIh+H3eCDNHhyuILTHLQbxgwmwmKtJI8BaSqWXXRE9qzaU8rm1U9lGkeLlURWOgFWP5HdjGKH8TPb+iGHm8eR6ExGu2m8mvK7sOSV5YWtVQmQfMGFndRBgLYz9IIsK/2UsUSxjFEHGShDIfRiCTKdRhgcBpsGD+mHTszOfGHS6GQumrIoDPa3FW7gK/pfD+eZcm0YY+8obruhvRvHEcgZy969GN1iihAD3wGximYJnGj6AYtXxoZ5aS02GHBK+wsHe/8ZAoaAhBRi0CjqoE/baosQcwNwbJCOJIDzCtCvGH2X3sk9fuSuVFBRd50fUELJ88UUTN5L2despwPQCrTjctpP7tH1Uiv29liIWPdG3qz8R3K5pPz9xf9LZfbvZ+93H/1cz8by72xRp2cmXVJ8tZVHv30agT/jvZGlcpgzsAk+XJ4ec7hffx1cv7BzmA34Xjr762Co3t5eXFZO4i/vh66q/Wuj466vV59xb03JblNj/devuSsTZqfch6Q1DbI0BKpP1G/5hjbnkh1B1JkCIISzcrcdCH3nMuvdoTuB61PsBvMteFXhwuUZ4laO8CUD/IQFmd0tAxxgIYW6YOAVAzvdSQY6LWa6Q5vV63M1278YjTUJmHA4ozcNVYy20orWsu7hFeK/iGH8rD9WUso9Gg31d/KSwNZRb4FtHhPQOI7+lF5DZSjiv6k8kqREl+rn9XaaCljdVRhpLgMkJ8MbWHYmm2fiNlwLOUw5GQWxeV32fos4dJqeREtGnuWqwLG5XGuDK5qHuH26ZaNM/ipLTJTTjINUrIBuUrydCseqwKo+SQeVevwSyHoqUhvxusFLZ9iUDOTSCtj0rnK10ashC9C+UBsS2EmA20uhe/WwQXcrHjDks8GzVHbXrfx6N4zY/o5KknRVnPN9e4Pml7RAmlsVUWJSPsttS47yPVXE70S45ok/nF71dDNkhtv7WIZF2I+17LhFmq25PoYTKY7QMYoxthvaCSCiTkuR1P0lzdUFhORW+I+7Ikfj+Btgll8bEfz0sw2TwtZq9UWSdocHku+aqwr0zobYpbRRmPocdkNoa8HMVcvG0PJy45NYDwG4yfARdQo8CdhlICRRxOmoXEaoQkbaGwWq5F6cJXIXgUsqyoN7YMfhxsBq/1kNJtttG6MTG0sbIhJst42Ao7szT8EmrI+NoYNDFdTSjco36eDAXvOjSErzuV/cC5DuSWWlirGAzIiKTZTjKAR24yv1evKysHwkKV3bnYYxtu6MZCp6x5IOYdEw11IqXk2SwNSA5iEAXwSjR2j2mwERZjLCX/FhKn6XA089LfO2XoKrYRWaDiKRwD4UqI/r6iKY9AJQblQLm1/L6sZWx/Lm214sn2rlbK+d2nmvEdtQOQtaAsI2ympanhN99u3tr6G54pxoZKlLwrLFxUJu5+p26YZUo6Zle2pRco+MWxTxOU0QKffcjNyWtYM9aP9mVM5gwWNynN6vluT4iLOA1pKZ7Jk8WwWDNqx/EcmMSHVfkxipvxwAYM2znjJBos4QoWyoQvV4TzlI1/6BcTaG6Zu2blrzs3JIvOCcByxv8WWIxaNTsJkIUFACM4VEVPQxA94tgky7vnnk+OTQ8oDx1L2HMTmbQO8v4XdUkoUkAPoD/U44bN0DsB97M+9+QBL7VkvyIcdc23dz5PL94BpigcGbtsUg/Hg8SJLG/YkuelvhfcBKMqdZB7wgnNnB8gZL3cA3gOEt8UM2qbswBZD4al8Sl2DBdbBMLlvhdFU4uY7+JGFmMi6dVtaCqA8PLBTN0o5ZngtcJcpCoyFfsfxF9r6mqXtZDELMCu+hVsMKysDqg1LggPxquwzBdLcmNS4Jb8cFL/Zu3UWzSmCJTHowaX3b8uhQy6u6VS0+/J2Q7eWIVOwMcPpGoIOHnr+PRg2aKrUuF/pzIsEcXD+uXt+JKbyfrg7ffn2NZvLuFcB2VcdB0SWEEw4MICUqKj3Z63mKRj1eDxfyIniB90b0r4gdwudbyAVDoxmjrufz69PT10yFYHncaO8J6KzvwoHVHqxFr/ldJ/hDPOdPpKg5GnocDIcZmDURnPcGYLT/JLSR2Odpx2ia7AIQiUB3Prq1CiWwKgbJRDpTvkPjoNkGOHMEWc+wH0f+PpYpWOOWAJUwTzB+FkaiZdmrjaleartJ2o6MZITaiaXTER39mLEypjADBCMB470/qDEOo3JhMDh58AV9dHF+dH15WX3/Eol1WMisV4wkqMH5CmYfF4jkbNxS3GTF4xW50nhhRXaWiCz49KQz0Y5TGvCN8iFZh17KreVQxLkAstnZkj0TxZvrmQy84sXWroQb50DSk1DgvrKxwOdF7+h/FhvGPMecwD1PvmnmqTISuvIR0ZCJfOkaDzaw9zWowQbpFlpgzebeaBk60JrVnHFtVaEsFSEU6e9ZBalSYeVJIi+Qu2W7W/s6XGvQ721xD7/eFrRaH3+tVWFMyExI7106BcZW9rPzTofKVreCkpkLm3tbrZIvORGkmXTmW4MBwRV10UxDUZqfnsFOagJEtYWWYwZoC1/f5QIricG0h+qWDkHYtQcNYNTlqwmea0OFrNyx/Cq4sixpRHxZGFWxw9duyyr2xubrXV91m6mrOt5xS5PR/9F96X1BWVrKNyszVV1n3RWe0JM4aSi8Er8DX4pYh2f8pUoIQ+3NjTGNQfLAD9hgiRm7/COwILPVp3phdUowRMIAr9LJ3hM03RRfzwOXkp3QtV2zMKSD3ao1tjwoBdnJq7OACCMSDx1DQ3IofTSiNM3lHIB4NMs0bsa+1u05dKCXzlW2Mc/jGDdgabqzjiWUu8nMWeYQxmunGiKXBXneq4Z0ES39UbFF57L/Lxs/qRS1lw3rFB9lXrGQqJcrbBoK7UYvVSJflVjBMBzXr7XtuTQr5SO5QSj7oBUr1LY6f3/BjFm7t17rhgr9ns/V4Rt0OsaEWb0XS++UOnVSrD1jvjUqbLKNR7IwV45O5DqmjaOxdRP7gRKai8XjNwyjfYbpCOJxFE2XzRw0gEsLdzTCIr84OW3e+3xNxmtKPk85bxQwkWGo0VEAUiybLDXTq3Jo81ZS7u73ZJ4KQc6xSb9UHy8uvokPl30rtCQ/NC9EgoSRt4gjh4SEFS8lqZYpH8fBXhsQJSNxjO09Qe4cUPtV65IKeXDK1uHTl+ebSGylgGyP5YqPz1q9FhGbJUJYSSekpJ203jpEVUkKOPoGka9czihY3DQ03EW/TOYzfxd3P3f+BLg/r9EnF+J/b323i8CHrx9/Yv4+vZ1E+O+M/lFDn4L0t03r/7afvVWNH77eHV22hKz4E6KD7BgjZriaAoWldylzZL4n+j5Yz8OVBXLGVNAdThEvmKIfOhH+Tt3eQY4q/zHDq8Td06OqwrBrvBk32JkArEXcP48bX2t9mAgOZ8mCc2Tps7VKCZZtboiAyiDI7aNd66WC6mT0qpjrFAYytn+v/almkrwmKfOgYaxpbs7UH9LABHLl1qDW5IgcN8irQHL/INXeyQvYpksHCjhE67QX4SaC4PtL/dcGdl4DaLRkg+NWuCxfqNGnsGyCms8dpo0ubmWNLCtaq6K6QZt42znaEopdVrBjaoB1v4ebobtb+nj/HJxVIVE5axSnAkWjd3TT13SQ7iCRMnyiFs/RvIJXTzFvMUkYhwuwF7NI3ZKA9dgri9Pv30sZIpyNsCIXDd5CbRPZVOvn583ZmDiEKy4fLfTnzRsteubRruJ7FyXKJ9j6A8MnwOhpfEXPVYGvuGIa8xwbcB5OEfVwrZY1BoeSLz0WparaLdiYWHTPM99X8Snh59OiCM4Dfwd2eUVZQZAeAt/ibPtf43l7bR+FYlVUkGJxnna2FDuC43k/JxXQuswiuIRnm8S5R4kQ3832+12KX6VaH+abQ3BAs9fBLvKXN3VTMCbxXJa2Ak5MWWT0snxanmq1xJNznsq1gJWVhleYJoFE979UVTXDxWeWyVylZhaoU4nM5Rwp/IWcEvEUhSA1KAOZu32o+7+afs7C2nGIS6lviuCqb/F54cxq3O36qgklXP7i0iWsMgd0YQonRhPS9GBnhFWhg5706orZEL4Qb4orsxudvQVszsBKIrFnDKC53gkxDDZzPB1SgA2culQiEUcTWLMGEHWKI5oAnsWsV6Z6cUsV1Cgdfz4VClwY836W83FDgNrUx43lrQtUWXw1RhX5Ut83yrcE/ZUcRdX06TI0nVzmVP+F7Lh51fiox5Kh4+zEy8ezR63KYwASPq63XwCMWG9WwzT7ZYAu7jZae+Pn75vin8TdonxIsESUBV+uZiMtkOslQbRncHTHCrXRoPDabpuWfUpms1UWAWXzXRkVx4tU6eN8YTGCEiF7XLjhBY+Fl8UYP1rzl0H5aXwX3KpdPBoRrrbePADe64MwfVk67Ya/tQwrZOyUz9RplbZaNhUm7yBVaVPBolmF926wSpIcvGj2H/aLd5SSNjjqBTyUjQWf4sGZv3fowHUJj7bNuoF6UxCi9tQGt5tN+tMlFiudxTq+FbBOi3B3VK6TUvkY1D3YHUxzGovEmcfY4YhbugE7dwRY2B3ZAacB/VSLhvgSU2KuxQYKNsR4UO5gYz7Q5YMjxEL8q9KgXzcWCa/qRSz0YFl7SeVCiaqsLh177JqNhXBFiXdYngzGShfq3P2gKUryh2YdKjYPEfKk0m+BvMwY/WpWTaZniHjfHIbqENzTZuazk3ckGHrGVCNMbGGB1yn2hc+y1Q8LosGTEdPw3u930etk3lnblrKO/r/z6l/iJdMHH8LK73c99X2DjC1qalCOuYQkjwspJ7imDKe/wADqVGUGSg/BIshoT1xTgcmu409DmHm0q6eo471oWki9h8U19hBEopHVuZUvpyD1qU/R0WlrFiwbtGRVaDi292UxllGeNX6KXWUansXdIvbZ7nef6UdVo9llyQ5G8Cs+PDpmkP1u2VjoL/1tPKMnI29Xa/fPMPb9d0qb5cbgxZAHMcs8wt57gfKTY96oc5VXzqvIBr8jty9YG1bDmiWPsJlfbLK6Nx94IKj2K/6zAC8NvHBrJr7xMfM+oqb9XTjzGmwmGkxvdK3kica4TNKuiaXLE9YNVdRNHVK+YctPMkw1aoALZD6yVrcd7/KIZ3Hq/OAdlDz4FG/mPbDuWD5Wbu0kTE/HxUJlqtEQzAnSkcAgwewHBkWa7ekxaeh8QFCAg8WT6bQTXt+d28FU82UIhfUeKQ5b8xnbB8QRm74rHPTIjF8JchNzjplK+bWnOxjbzieqLJK0OoT0T2VLgq0fzSnLcvyon2tw4x2CzWQlzIU2a25XufnKsmncihaCRGWjkImR1lc/naHKzMFmDcIKbvmUXHRk/dYdG0zK1fmvL6NmkYMr2nbjl2OaS9jPoRW0WV5oeI4Nex5mRjG8WF/NC2heEdf+3sv0+GUtKHKnuuIq3hpu2rooEDRIDFKhtTu0B9OMbOdzlwhh0+zVXz9A+bGyPALsZQxcZeFo+L0v4Oy2KbPPfEhnYhhmNcT2Xi9HqdlNnMfZ1Mu5D7UpgKjnUKmL+MgkQKr73Hzh0qUpAPJ8i9VCsYbzE3QHVM8qAJEGe64IEnkUyZkXthu1s0b5aQxvR+tfsbVo4/aqsuTMQ4O43J2sW/6XhilSiQzKReN/ba9VRGgsymwUWJA5UwhGwutQtysAd+lUvFaEaC56YHUELnsuFVTh9TPmLdrNUCfNn+haUKnzqmpYsyRsv9aX3UzpmJnlgq6D9YssIAFyhPEeAsmenlurHG5vHw1flW3jNCrssJwbuN3X2DY+HRDA9yFIe5HLlzOLXy1qxcuu8aCuBzxHYHgvtMrL9f6Q/dSz0WKBtTSH2Qec9u76EUZCpTKop7Ril0uZYrHJsokCejQmFAfCRoz0tQSz+Q6Puusht84idZcxauFbWET1B9a6TDCx97cPGjOwWq7ooFnQqujod2hj1qm++vemzqmazxi3+TqFWfvmrS3LLhX8Yiqkiq477uSwqdvbr7LgtlIvH+PWwEMg3MYzeegawzjEsZFByvzkc/KQNNLYOMl+9DDbGYnOA7jcaUaPPNA3WKNV2Z20L2HMAwr5Smjnt9xLx/bL9++AUR/7H4+ss+gSzzaQ4ELcvjLmw50s5Q1fog1jKdFXXTO0lYF3Dri2MZRklGqq6oqz9txTJx7D1ZEnDG3tTPs3PNwzF0StCmDDnBUH5davFYPh//g8hh8BgyWv4LriHXV9wjr2a+EH245HpsNV0C4l/FyDGt2y6R29Pdtgzf2jvy/HPybbxm8wdfzEbdr7abBLcyjQO+1WloIe6G+UCQ+Xx6eFdtAKJUFxRlO4/PPx92jTvHpi0Tg5rO8yCBK05kM5dA+tbgO6QAjKMoUN1UiAqYPtNRk4IbZyMfPHlsjMkvTFrNcrN3aoQ1LKtAHTXjirwbgnjFt1K3t32rmB6ZjxRe541czuvApZt95oZz4tIUhTXQa+p30/injyFVHxz4V2wUL2dmvlMsXRcXjW/sEzSz2EhJgFJVoGItY9Qrbp08ENUsi+VVbnPpZCAsYJZPVtiv2+FB8tgjJquP7Kw7SWrXy1t/H72Txt2YN98K4TtngwsX2lDa2H/PhqKgQhoMoiaV+/4to2PZAJa8Bh2lvV/tE7i3ghpY6AcrchPbp5FN3xeY0awvbIBujbj7Yt1DNSdEDOo4FvQGuJGfcyuUthiqsVHo+XiTO5/x9jgPaZNm3zgtxfx2Uv4cUEgbUYUouUwtLHdCf6sZLfeFOSi5g7BYAOA++9WO6xkCZk7l5tY8Tm73Zv9Xw1CTfUS/sa6HoM0b+dbvuGn/447N8LlIFE0SZFbiwyLdqpDWo5EFSA8Yw+X5TGNFlg2zoZd9ONUykpF2vrjGsoBKDS3ILmtgV+97e3h7+3+RxZyWeI6CrGj//3N5ribn/tbGHPxp5m7v0cB8fKhnYbJJtrIRgXfsKq9CDgVO8+5OYzaixIeXw+lO+jxxGD3WSx7yATxpYdMeSWbRTGlOqV3RN8riUMORI4TavtaG/aoWNgtbVakTRTk7PteWJnp2cmmvLK2p2DFqur5NvpTGTeuprPa1g3DXBSlTFlUVynZ9BWSUd8ajRxak+mOmjEdJp740pwUf8F3/uSJemobvPIM5htZUhcNsKgcVxTg3TAbBXxeDTlyEc9uvFSHmKO4qScnzwQZCtOGHQ5eOiinfBbFaeXYo8fLLOOudF0YX6JGINoTTPbB5owgvTyxBMfk8W2Xf4RbBVvi50ydf7uuhtva/LOWg+WrBuZIWxqFx/z7T3xA2Ajyd9jCQgpzRcPB9mQ1zR592P0AVZbJJnN5IZxzJ9/UYkasVxZ6Uo1Yrzzsou2RzjrXKoyw6vqHQbh9OoXK3edbQ6Y3a138hy7hDm0JWk86dyd9Jz7XjR89H5VAQPy06mN23O7FlWXKAGTrFDOyus4p6roLeV47TU49s2n2tuxEwx9BLF+FVG4xMeRJZnRb3cdteGsTDFgjVRr2duhzPmSDlDWH8tjWg0i6KF3oZrhKkdm2858Z/39nPASqB7FaYRHZiDZG+ACSde0m/+ji28aVYTREawrsdDiZPiywB4PUwxLcU6zcDpbV6dWUJkKMIsr1wn5eefAShRDz+Hp/zhRupq1edNYX4HnWsGlg8Ql+juUxPqUjDKhW8UkW5VgLrGkb0ij7wwW1xecHUWoEKCleGA3eVMcqD+1u0U/fNGaeaG/rFu0P3lh8viE4Jw00jMPXn8fR/auVNuuH3vzzL8voB7Wc1fxTHbXznWDdL8a7tRNKpb+RmsTxHGcoEaF2ip7l77TVn+uOsZPO/+8AS6wI15AWXQLDEefS9e7rlsEX2tspMrhtgJtNsRX8AQ1B8oKVLGzQN36sJAJclRCE2QSJzDXhGMpLXUtihJWTsBpeNECzCb0VOHkFgn/xQfmmZRWZGQ9W46WqaD3n/7U1Xrr0kKlz/tqQg99q/OpMHDcdjjR+sBO6u9nF2yohd4rxTEybFux0yfe0Y7AF1ntYlRfql2YOG7s3fkNm8+q0fOpvqoPe2HOtEKtax43P67TLZVJKd6ZhedVLh9HokGeuDfq3yQ5vbzxmx8QFiP3XlK01P+ufIAj5H0QxllyWypde2z+uSPl59GpQ+Y44j5yxbHoqE+ed6k8asGje9F8OCPT3qq+Jk+NTahA1k3x4LF1iWz73nC8gVzsz3hzMmW46pwu5JJ4zrJkoyaPB+nniwVM9A6BIm/AqOSHA84z7lsgrUERi4OGvig1cS91mAFhq7DHVL2B5atEjUIfThmaiFlrWVFeVH6LDNByeOcr1PJSri+PP1Go6xUIk88cJyw5Nax67tYzWRaGA7h3/H+Xj5evaPgqBimnoTGTsta/4jKHHDsqX2e9q3fcaiSTEytVqGLOv4G/uc48AYN8/xspUpopTKYFTPugWHAs+rVKGubMlRp2ZbQV50h/kJ8quSo8pc+fPxQCZlvmOnGYUBBSRu86QE/CqRmPYhFbevyzLf7eJ79qMrk38BOi5Oo6k1FWiqY9mX5MMqx1a4zJw/zArGXNQVz2m3uzTU9uX6qwo/6UfnoHLui5dHN6xpPV1dXnt28It3zDsWV9ZSHN69H9xvUyz29eU31pKWCcGuqP8PpW3b41sx3N6msnli3ldszEwj38iOJQah76vC7OQgG9eEq9REK/sCXcSxg+atb+GUJ3iqJ7KYMQ7YUi9MuDQtVO5lK3/QyulNAK7V7YB3KqaWi7SpRp5Wila3eK3/Hb3I5iMAyO8FvycfZwox9rrInCrN3tK/HpCzUQDfFH07LEmhD9KZZStIVE+HKElJBt0IuRuMxur6VMYJfWyBz1fPYGe4RXTwvd4SXKYbHPD/9X+XxWps="
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
